# Customer Sales Insights Mini-Project
### Superstore Dataset — Subqueries, CTEs & Window Functions in SQL

**Dataset:** Sample Superstore (9,994 order-line rows, 793 customers, 1,862 products)
**Engine:** SQLite via Python's `sqlite3` (all SQL used is ANSI-standard — CTEs and
window functions run unchanged on PostgreSQL / MySQL 8+ / SQL Server).

**Workflow**
1. Load raw CSV into a staging table `superstore_raw`
2. Build normalized `customers`, `products`, `orders` tables with `SELECT DISTINCT`
3. Subqueries — above-average sales, highest order per customer
4. CTEs — per-customer aggregation
5. Window functions — `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`
6. JOIN + CTE + Window Function combined final report
7. Business questions: top customers, low customers, single-order customers, above-average customers


In [1]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

df = pd.read_csv('superstore.csv')
print(df.shape)
df.head()

(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


## Step 1 — Load into staging table `superstore_raw`

The CSV is loaded as-is into a single staging table, exactly mirroring the raw
source. No cleaning or normalization happens yet at this stage.

In [2]:
conn = sqlite3.connect('superstore.db')
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

pd.read_sql_query("SELECT COUNT(*) AS staging_row_count FROM superstore_raw;", conn)

,staging_row_count
0,9994


## Step 2 — Create normalized tables: `customers`, `products`, `orders`

**Design note:** the same `Customer ID` in this dataset ships to more than one
City/State/Region across different orders (780 of 793 customers have multiple
ship-to cities). Location is therefore modeled as an attribute of the
**order** (the shipment), not the **customer**. Similarly, 32 `Product ID`s
carry two slightly different spellings of `Product Name` in the raw data — a
known quirk of this dataset — so `MIN(Product Name)` is used to deterministically
pick one canonical name per product.

In [3]:
conn.executescript('''
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id    TEXT PRIMARY KEY,
    customer_name  TEXT,
    segment        TEXT,
    country        TEXT
);

CREATE TABLE products (
    product_id     TEXT PRIMARY KEY,
    category       TEXT,
    sub_category   TEXT,
    product_name   TEXT
);

CREATE TABLE orders (
    row_id         INTEGER PRIMARY KEY,
    order_id       TEXT,
    order_date     TEXT,
    ship_date      TEXT,
    ship_mode      TEXT,
    customer_id    TEXT,
    product_id     TEXT,
    city           TEXT,
    state          TEXT,
    region         TEXT,
    sales          REAL,
    quantity       REAL,
    discount       REAL,
    profit         REAL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id)  REFERENCES products(product_id)
);
''')
print('Tables created')

Tables created


## Step 3 — Populate tables using `SELECT DISTINCT`

In [4]:
conn.execute('''
INSERT INTO customers (customer_id, customer_name, segment, country)
SELECT DISTINCT "Customer ID", "Customer Name", "Segment", "Country"
FROM superstore_raw;
''')

conn.execute('''
INSERT INTO products (product_id, category, sub_category, product_name)
SELECT "Product ID", "Category", "Sub-Category", MIN("Product Name")
FROM superstore_raw
GROUP BY "Product ID", "Category", "Sub-Category";
''')

conn.execute('''
INSERT INTO orders (row_id, order_id, order_date, ship_date, ship_mode,
                     customer_id, product_id, city, state, region,
                     sales, quantity, discount, profit)
SELECT DISTINCT "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode",
       "Customer ID", "Product ID", "City", "State", "Region",
       "Sales", "Quantity", "Discount", "Profit"
FROM superstore_raw;
''')
conn.commit()

pd.read_sql_query('''
SELECT (SELECT COUNT(*) FROM customers) AS n_customers,
       (SELECT COUNT(*) FROM products)  AS n_products,
       (SELECT COUNT(*) FROM orders)    AS n_orders
''', conn)

,n_customers,n_products,n_orders
0,793,1862,9994


## Step 4 — Subqueries

**4a. Order lines with sales above the overall average sale amount**

In [5]:
pd.read_sql_query('''
SELECT row_id, order_id, customer_id, product_id, sales
FROM orders
WHERE sales > (SELECT AVG(sales) FROM orders)
ORDER BY sales DESC
LIMIT 10;
''', conn)

,row_id,order_id,customer_id,product_id,sales
0,2698,CA-2015-145317,SM-20320,TEC-MA-10002412,22638.480
1,6827,CA-2017-118689,TC-20980,TEC-CO-10004722,17499.950
2,8154,CA-2018-140151,RB-19360,TEC-CO-10004722,13999.960
3,2624,CA-2018-127180,TA-21385,TEC-CO-10004722,11199.968
4,4191,CA-2018-166709,HL-15040,TEC-CO-10004722,10499.970
5,9040,CA-2017-117121,AB-10105,OFF-BI-10000545,9892.740
6,4099,CA-2015-116904,SC-20095,OFF-BI-10001120,9449.950
7,4278,US-2017-107440,BS-11365,TEC-MA-10001047,9099.930
8,8489,CA-2017-158841,SE-20110,TEC-MA-10001127,8749.950
9,6426,CA-2017-143714,CC-12370,TEC-CO-10004722,8399.976


**4b. Each customer's single highest-value order (correlated subquery)**

In [6]:
pd.read_sql_query('''
SELECT o.customer_id, o.order_id, o.order_total
FROM (
    SELECT customer_id, order_id, SUM(sales) AS order_total
    FROM orders GROUP BY customer_id, order_id
) o
WHERE o.order_total = (
    SELECT MAX(o2.order_total)
    FROM (SELECT customer_id, order_id, SUM(sales) AS order_total
          FROM orders GROUP BY customer_id, order_id) o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.order_total DESC
LIMIT 10;
''', conn)

,customer_id,order_id,order_total
0,SM-20320,CA-2015-145317,23661.228
1,TC-20980,CA-2017-118689,18336.740
2,RB-19360,CA-2018-140151,14052.480
3,TA-21385,CA-2018-127180,13716.458
4,BM-11140,CA-2015-139892,10539.896
5,HL-15040,CA-2018-166709,10499.970
6,SC-20095,CA-2015-116904,9900.190
7,AB-10105,CA-2017-117121,9892.740
8,BS-11365,US-2017-107440,9135.190
9,SE-20110,CA-2017-158841,8805.040


**4c. Customers whose total lifetime sales exceed the average total sales across all customers**

In [7]:
pd.read_sql_query('''
SELECT customer_id, total_sales
FROM (SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id) cust_totals
WHERE total_sales > (
    SELECT AVG(total_sales) FROM (
        SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id
    )
)
ORDER BY total_sales DESC
LIMIT 10;
''', conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
5,KL-16645,14175.229
6,SC-20095,14142.334
7,HL-15040,12873.298
8,SE-20110,12209.438
9,CC-12370,12129.072


## Step 5 — CTE: total sales per customer

In [8]:
pd.read_sql_query('''
WITH customer_sales AS (
    SELECT customer_id,
           COUNT(DISTINCT order_id) AS num_orders,
           SUM(sales)  AS total_sales,
           SUM(profit) AS total_profit,
           AVG(sales)  AS avg_line_sales
    FROM orders
    GROUP BY customer_id
)
SELECT * FROM customer_sales ORDER BY total_sales DESC LIMIT 10;
''', conn)

,customer_id,num_orders,total_sales,total_profit,avg_line_sales
0,SM-20320,5,25043.050,-1980.7393,1669.536667
1,TC-20980,5,19052.218,8981.3239,1587.684833
2,RB-19360,6,15117.339,6976.0959,839.852167
3,TA-21385,4,14595.620,4703.7883,1459.562000
4,AB-10105,10,14473.571,5444.8055,723.678550
5,KL-16645,12,14175.229,806.8550,488.801000
6,SC-20095,9,14142.334,5757.4119,642.833364
7,HL-15040,6,12873.298,5622.4292,1170.299818
8,SE-20110,11,12209.438,2650.6769,642.602000
9,CC-12370,5,12129.072,2177.0493,1102.642909


## Step 6 — Window functions

**6a. `ROW_NUMBER`, `RANK`, `DENSE_RANK` of customers by total sales**

In [9]:
pd.read_sql_query('''
WITH customer_sales AS (SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT customer_id, total_sales,
       ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS row_num,
       RANK()       OVER (ORDER BY total_sales DESC) AS sales_rank,
       DENSE_RANK() OVER (ORDER BY total_sales DESC) AS sales_dense_rank
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 10;
''', conn)

,customer_id,total_sales,row_num,sales_rank,sales_dense_rank
0,SM-20320,25043.050,1,1,1
1,TC-20980,19052.218,2,2,2
2,RB-19360,15117.339,3,3,3
3,TA-21385,14595.620,4,4,4
4,AB-10105,14473.571,5,5,5
5,KL-16645,14175.229,6,6,6
6,SC-20095,14142.334,7,7,7
7,HL-15040,12873.298,8,8,8
8,SE-20110,12209.438,9,9,9
9,CC-12370,12129.072,10,10,10


**6b. Rank each customer's own orders by value (`PARTITION BY`)**

In [10]:
pd.read_sql_query('''
WITH order_totals AS (SELECT customer_id, order_id, SUM(sales) AS order_total FROM orders GROUP BY customer_id, order_id)
SELECT customer_id, order_id, order_total,
       RANK() OVER (PARTITION BY customer_id ORDER BY order_total DESC) AS rank_within_customer
FROM order_totals
ORDER BY customer_id, rank_within_customer
LIMIT 15;
''', conn)

,customer_id,order_id,order_total,rank_within_customer
0,AA-10315,CA-2017-103982,4406.072,1
1,AA-10315,CA-2015-128055,726.548,2
2,AA-10315,CA-2018-147039,374.480,3
3,AA-10315,CA-2015-138100,29.500,4
4,AA-10315,CA-2016-121391,26.960,5
5,AA-10375,CA-2017-131065,513.520,1
6,AA-10375,CA-2016-140921,178.370,2
7,AA-10375,CA-2018-100230,149.872,3
8,AA-10375,CA-2016-114503,84.960,4
9,AA-10375,US-2018-169488,56.860,5


**6c. Segment customers into sales quartiles with `NTILE(4)`**

In [11]:
pd.read_sql_query('''
WITH customer_sales AS (SELECT customer_id, SUM(sales) AS total_sales FROM orders GROUP BY customer_id)
SELECT customer_id, total_sales, NTILE(4) OVER (ORDER BY total_sales DESC) AS sales_quartile
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 10;
''', conn)

,customer_id,total_sales,sales_quartile
0,SM-20320,25043.050,1
1,TC-20980,19052.218,1
2,RB-19360,15117.339,1
3,TA-21385,14595.620,1
4,AB-10105,14473.571,1
5,KL-16645,14175.229,1
6,SC-20095,14142.334,1
7,HL-15040,12873.298,1
8,SE-20110,12209.438,1
9,CC-12370,12129.072,1


## Step 7 — JOIN + CTE + Window Function combined
Final report: **customer, total sales, rank**

In [12]:
final_report = pd.read_sql_query('''
WITH customer_sales AS (
    SELECT o.customer_id, SUM(o.sales) AS total_sales, SUM(o.profit) AS total_profit,
           COUNT(DISTINCT o.order_id) AS num_orders
    FROM orders o
    GROUP BY o.customer_id
),
ranked_customers AS (
    SELECT cs.customer_id, c.customer_name, c.segment,
           cs.total_sales, cs.total_profit, cs.num_orders,
           ROW_NUMBER() OVER (ORDER BY cs.total_sales DESC) AS sales_rank
    FROM customer_sales cs
    JOIN customers c ON c.customer_id = cs.customer_id
)
SELECT * FROM ranked_customers ORDER BY sales_rank;
''', conn)
final_report.head(15)

,customer_id,customer_name,segment,total_sales,total_profit,num_orders,sales_rank
0,SM-20320,Sean Miller,Home Office,25043.050,-1980.7393,5,1
1,TC-20980,Tamara Chand,Corporate,19052.218,8981.3239,5,2
2,RB-19360,Raymond Buch,Consumer,15117.339,6976.0959,6,3
3,TA-21385,Tom Ashbrook,Home Office,14595.620,4703.7883,4,4
4,AB-10105,Adrian Barton,Consumer,14473.571,5444.8055,10,5
5,KL-16645,Ken Lonsdale,Consumer,14175.229,806.8550,12,6
6,SC-20095,Sanjit Chand,Consumer,14142.334,5757.4119,9,7
7,HL-15040,Hunter Lopez,Consumer,12873.298,5622.4292,6,8
8,SE-20110,Sanjit Engle,Consumer,12209.438,2650.6769,11,9
9,CC-12370,Christopher Conant,Consumer,12129.072,2177.0493,5,10


## Step 8 — Business Queries

**8a. Top 10 customers by total sales**

In [13]:
top10 = pd.read_sql_query('''
WITH customer_sales AS (SELECT o.customer_id, SUM(o.sales) AS total_sales FROM orders o GROUP BY o.customer_id)
SELECT c.customer_name, c.segment, cs.total_sales,
       RANK() OVER (ORDER BY cs.total_sales DESC) AS rank
FROM customer_sales cs JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales DESC LIMIT 10;
''', conn)
top10

,customer_name,segment,total_sales,rank
0,Sean Miller,Home Office,25043.050,1
1,Tamara Chand,Corporate,19052.218,2
2,Raymond Buch,Consumer,15117.339,3
3,Tom Ashbrook,Home Office,14595.620,4
4,Adrian Barton,Consumer,14473.571,5
5,Ken Lonsdale,Consumer,14175.229,6
6,Sanjit Chand,Consumer,14142.334,7
7,Hunter Lopez,Consumer,12873.298,8
8,Sanjit Engle,Consumer,12209.438,9
9,Christopher Conant,Consumer,12129.072,10


**8b. Bottom 10 (lowest-spending) customers by total sales**

In [14]:
bottom10 = pd.read_sql_query('''
WITH customer_sales AS (SELECT o.customer_id, SUM(o.sales) AS total_sales FROM orders o GROUP BY o.customer_id)
SELECT c.customer_name, c.segment, cs.total_sales,
       RANK() OVER (ORDER BY cs.total_sales ASC) AS rank_from_bottom
FROM customer_sales cs JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY cs.total_sales ASC LIMIT 10;
''', conn)
bottom10

,customer_name,segment,total_sales,rank_from_bottom
0,Thais Sissman,Consumer,4.833,1
1,Lela Donovan,Corporate,5.304,2
2,Carl Jackson,Corporate,16.520,3
3,Mitch Gastineau,Corporate,16.739,4
4,Roy Skaria,Home Office,22.328,5
5,Susan Gilcrest,Corporate,47.946,6
6,Ricardo Emerson,Consumer,48.360,7
7,Larry Blacks,Consumer,50.188,8
8,Adrian Shami,Home Office,58.820,9
9,Jasper Cacioppo,Consumer,71.263,10


**8c. Single-order customers (placed exactly one order)**

In [15]:
single_order = pd.read_sql_query('''
WITH customer_orders AS (SELECT customer_id, COUNT(DISTINCT order_id) AS num_orders FROM orders GROUP BY customer_id)
SELECT c.customer_name, c.segment, co.num_orders
FROM customer_orders co JOIN customers c ON c.customer_id = co.customer_id
WHERE co.num_orders = 1
ORDER BY c.customer_name;
''', conn)
print(f"{len(single_order)} single-order customers out of 793 total")
single_order

12 single-order customers out of 793 total


,customer_name,segment,num_orders
0,Anemone Ratner,Consumer,1
1,Anthony O'Donnell,Corporate,1
2,Carl Jackson,Corporate,1
3,Jenna Caffey,Consumer,1
4,Jocasta Rupert,Consumer,1
5,Lela Donovan,Corporate,1
6,Mitch Gastineau,Corporate,1
7,Patricia Hirasaki,Home Office,1
8,Ricardo Emerson,Consumer,1
9,Roland Murray,Consumer,1


**8d. Customers with above-average total sales**

In [16]:
above_avg = pd.read_sql_query('''
WITH customer_sales AS (SELECT o.customer_id, SUM(o.sales) AS total_sales FROM orders o GROUP BY o.customer_id)
SELECT c.customer_name, c.segment, cs.total_sales,
       ROUND(cs.total_sales - (SELECT AVG(total_sales) FROM customer_sales), 2) AS diff_from_avg
FROM customer_sales cs JOIN customers c ON c.customer_id = cs.customer_id
WHERE cs.total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY cs.total_sales DESC;
''', conn)
print(f"{len(above_avg)} of 793 customers ({len(above_avg)/793:.0%}) are above the average customer lifetime sales of $2,896.85")
above_avg.head(10)

294 of 793 customers (37%) are above the average customer lifetime sales of $2,896.85


,customer_name,segment,total_sales,diff_from_avg
0,Sean Miller,Home Office,25043.050,22146.20
1,Tamara Chand,Corporate,19052.218,16155.37
2,Raymond Buch,Consumer,15117.339,12220.49
3,Tom Ashbrook,Home Office,14595.620,11698.77
4,Adrian Barton,Consumer,14473.571,11576.72
5,Ken Lonsdale,Consumer,14175.229,11278.38
6,Sanjit Chand,Consumer,14142.334,11245.49
7,Hunter Lopez,Consumer,12873.298,9976.45
8,Sanjit Engle,Consumer,12209.438,9312.59
9,Christopher Conant,Consumer,12129.072,9232.22


**8e. Sales distribution across customer segments (rank within segment)**

In [17]:
pd.read_sql_query('''
WITH customer_sales AS (SELECT o.customer_id, SUM(o.sales) AS total_sales FROM orders o GROUP BY o.customer_id)
SELECT c.segment, c.customer_name, cs.total_sales,
       RANK() OVER (PARTITION BY c.segment ORDER BY cs.total_sales DESC) AS rank_in_segment
FROM customer_sales cs JOIN customers c ON c.customer_id = cs.customer_id
ORDER BY c.segment, rank_in_segment
LIMIT 15;
''', conn)

,segment,customer_name,total_sales,rank_in_segment
0,Consumer,Raymond Buch,15117.339,1
1,Consumer,Adrian Barton,14473.571,2
2,Consumer,Ken Lonsdale,14175.229,3
3,Consumer,Sanjit Chand,14142.334,4
4,Consumer,Hunter Lopez,12873.298,5
5,Consumer,Sanjit Engle,12209.438,6
6,Consumer,Christopher Conant,12129.072,7
7,Consumer,Greg Tran,11820.120,8
8,Consumer,Becky Martin,11789.630,9
9,Consumer,Seth Vernon,11470.950,10


## Key Insights

1. **Revenue is concentrated at the top.** The top 10 customers by lifetime
   sales (1.3% of the 793-customer base) each spent well above the average
   customer total of **$2,896.85** — the single biggest spender, Sean Miller,
   totals **$25,043** across just 5 orders.
2. **294 of 793 customers (37%) sit above the average lifetime-sales bar** —
   confirming a long right tail: most customers spend modestly, while a
   minority drive a disproportionate share of the **$2.30M** in total sales.
3. **High sales doesn't guarantee high profit.** The #1 customer by sales
   (Sean Miller) is actually **unprofitable overall (-$1,980)**, and several
   other high-revenue accounts (e.g. Becky Martin) also carry negative total
   profit — likely tied to heavy discounting. Sales rank and profit rank are
   not interchangeable signals for account prioritization.
4. **Only 12 customers (1.5%) are single-order, one-and-done buyers** — the
   large majority of the customer base is repeat business, which is a
   healthy retention signal for a retail operation.
5. **Consumer is the largest segment by revenue** (~$1.16M), roughly 1.6x
   Corporate (~$0.71M) and 2.7x Home Office (~$0.43M), so segment-level
   promotions have the widest reach when targeted at Consumer.
6. **West region leads in both sales and order volume** (~$725K / 1,611
   orders), followed by East, Central, and South — a useful lens for
   regional inventory and logistics planning.
